# 🫁 Chest X-Ray Tiered Classifier — Fully Automated Colab Training Notebook

This notebook **runs from scratch** and automates every step:

1. Clones the GitHub repo (`xdboi123yes/xray`)
2. Installs dependencies (compatible with the Colab base image, with **automatic kernel restart**)
3. Downloads the NIH ChestX-ray14 dataset from Kaggle
4. Preprocesses + splits the data (train/val/test/calibration)
5. Downloads the Ark+ checkpoint (fallback: Swin-Base ImageNet)
6. Trains Tier 1 (MobileNetV2) + Tier 2 (EffNet-B4) + Tier 2 (Ark+)
7. Calibration (temperature scaling + ECE)
8. Ablation matrix (A8–A15) + honest `ablation.json`
9. Statistical tests (DeLong + Bootstrap)
10. ONNX + INT8 export
11. Syncs to Drive **and** downloads a single ZIP

**Usage:** Run cells top-to-bottom with `Shift+Enter`. The only manual step is uploading `kaggle.json` in cell #2.

**GPU required:** `Runtime > Change runtime type > T4 GPU` (or A100/L4).

## 0. Configuration (single source of truth)

In [ ]:
# ============================================================================
# CONFIGURATION — all switches live here
# ============================================================================
GITHUB_REPO_URL  = 'https://github.com/xdboi123yes/xray.git'
GITHUB_BRANCH    = 'main'
PROJECT_ROOT     = '/content/xray'                      # clone target
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/xray_outputs' # output mirror
USE_DRIVE        = True   # set False to skip Drive mount

# Training switches — flip any to False to skip that tier
RUN_TIER1         = True   # MobileNetV2
RUN_TIER2_EFFNET  = True   # EfficientNet-B4
RUN_TIER2_ARK     = True   # Ark+ Swin
RUN_CALIBRATION   = True
RUN_ABLATION      = True
RUN_STAT_TESTS    = True
RUN_ONNX_EXPORT   = True

# Quick dry-run mode (1 epoch per tier, for validation)
DRY_RUN = False

# Kaggle dataset slug — NIH ChestX-ray14 (Kaggle mirror)
KAGGLE_DATASET = 'nih-chest-xrays/data'

print('Configuration loaded.')
print(f'  Repo:   {GITHUB_REPO_URL} ({GITHUB_BRANCH})')
print(f'  Train:  Tier1={RUN_TIER1}  Tier2-Eff={RUN_TIER2_EFFNET}  Tier2-Ark={RUN_TIER2_ARK}')
print(f'  Drive:  {USE_DRIVE} → {DRIVE_OUTPUT_DIR}')
print(f'  DryRun: {DRY_RUN}')

## 1. Upload Kaggle credentials (one-time manual step)

Drag-and-drop your `kaggle.json` (Kaggle Account > API > Create New API Token).

In [ ]:
import os, json
from pathlib import Path

KAGGLE_CONFIG_DIR = Path.home() / '.kaggle'
KAGGLE_CONFIG_DIR.mkdir(exist_ok=True)
KAGGLE_JSON_PATH = KAGGLE_CONFIG_DIR / 'kaggle.json'

if KAGGLE_JSON_PATH.exists():
    print(f'✅ Kaggle credentials already present: {KAGGLE_JSON_PATH}')
else:
    from google.colab import files
    print('📤 Upload your kaggle.json (drag-and-drop):')
    uploaded = files.upload()
    fname = next(iter(uploaded.keys()))
    if fname != 'kaggle.json':
        # rename if needed
        os.rename(fname, 'kaggle.json')
        fname = 'kaggle.json'
    Path(fname).rename(KAGGLE_JSON_PATH)
    KAGGLE_JSON_PATH.chmod(0o600)
    # sanity check
    with open(KAGGLE_JSON_PATH) as f:
        creds = json.load(f)
    assert 'username' in creds and 'key' in creds, 'kaggle.json is malformed'
    print(f'✅ Kaggle credentials installed: user={creds["username"]}')

## 2. Mount Drive (for output sync)

In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    print(f'✅ Drive mounted. Output mirror: {DRIVE_OUTPUT_DIR}')
else:
    print('⏭️  Drive skipped (USE_DRIVE=False).')

## 3. Clone from GitHub

In [ ]:
import subprocess

if os.path.isdir(PROJECT_ROOT):
    print(f'📂 Repo already exists: {PROJECT_ROOT}. Pulling latest...')
    subprocess.check_call(['git', '-C', PROJECT_ROOT, 'fetch', 'origin'])
    subprocess.check_call(['git', '-C', PROJECT_ROOT, 'checkout', GITHUB_BRANCH])
    subprocess.check_call(['git', '-C', PROJECT_ROOT, 'reset', '--hard', f'origin/{GITHUB_BRANCH}'])
else:
    subprocess.check_call(['git', 'clone', '--branch', GITHUB_BRANCH, '--depth', '1', GITHUB_REPO_URL, PROJECT_ROOT])
    print(f'✅ Clone complete: {PROJECT_ROOT}')

os.chdir(PROJECT_ROOT)
print(f'📍 cwd: {os.getcwd()}')
print()
print(subprocess.check_output(['git', 'log', '-1', '--oneline']).decode())

## 4. Install dependencies (Colab-compatible) + automatic kernel restart

**IMPORTANT:** This cell restarts the kernel once installation completes. After the restart,
continue from the next cell. On a second run, the `flag` file is present so installation is skipped.

In [ ]:
import os, sys, subprocess

INSTALL_FLAG = '/content/.deps_installed'

if os.path.exists(INSTALL_FLAG):
    print('✅ Dependencies already installed (flag present). Skipping.')
else:
    print('📦 Installing dependencies (3-5 min)...')
    # First, pin the packages the Colab base image clashes with
    # Force-upgrade tqdm/packaging to training-compatible versions
    
    # 1) Bump tqdm/packaging on the Colab base image
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--quiet',
        '--upgrade', 'tqdm>=4.66.3', 'packaging>=24.0'
    ])
    
    # 2) Install repo requirements
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--quiet',
        '-r', 'requirements.txt'
    ])
    
    # 3) Training extras (if present)
    if os.path.exists('requirements-training.txt'):
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '--quiet',
            '-r', 'requirements-training.txt'
        ])
    
    # 4) Colab-specific extras
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--quiet',
        'opencv-python-headless', 'kaggle'
    ])
    
    # Drop a flag so we don't reinstall after the restart
    Path(INSTALL_FLAG).touch()
    print('✅ All dependencies installed.')
    print()
    print('🔄 Restarting the kernel — this is expected.')
    print('   After the restart, CONTINUE from the next cell (do not re-run the cells above).')
    print()
    # Automatic restart
    import IPython
    IPython.Application.instance().kernel.do_shutdown(restart=True)

## 5. Post-restart: restore configuration and cwd

Continue from this cell after the kernel restart.

In [ ]:
import os, sys, subprocess
from pathlib import Path

# Re-declare the configuration (lost on kernel restart)
GITHUB_REPO_URL  = 'https://github.com/xdboi123yes/xray.git'
GITHUB_BRANCH    = 'main'
PROJECT_ROOT     = '/content/xray'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/xray_outputs'
USE_DRIVE        = True
RUN_TIER1         = True
RUN_TIER2_EFFNET  = True
RUN_TIER2_ARK     = True
RUN_CALIBRATION   = True
RUN_ABLATION      = True
RUN_STAT_TESTS    = True
RUN_ONNX_EXPORT   = True
DRY_RUN = False
KAGGLE_DATASET = 'nih-chest-xrays/data'

os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Remount Drive (also lost on restart)
if USE_DRIVE and not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

import torch
print(f'📍 cwd: {os.getcwd()}')
print(f'🐍 Python: {sys.version.split()[0]}')
print(f'🔥 PyTorch: {torch.__version__}')
print(f'⚡ CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('   ⚠️  No GPU detected. Runtime > Change runtime type > T4 GPU')

## 6. Download NIH dataset from Kaggle and merge

**Size:** ~45 GB download + extract. If `MyDrive/nih_data/images/` already exists in Drive, it's used directly.

In [ ]:
import os, shutil, glob, subprocess

DRIVE_DATA   = '/content/drive/MyDrive/nih_data'
COLAB_DATA   = '/content/nih_data'
MERGED_DIR   = f'{COLAB_DATA}/images'        # flat folder on Colab disk
DRIVE_MERGED = f'{DRIVE_DATA}/images'         # mirror in Drive
CSV_NAME     = 'Data_Entry_2017.csv'
MIN_IMAGES   = 100000

os.makedirs(COLAB_DATA, exist_ok=True)

def count_files(d):
    if not os.path.isdir(d): return 0
    return len([f for f in os.listdir(d) if os.path.isfile(os.path.join(d, f))])

def merge_subdirs(base_dir, target_dir):
    """Merge images_001..012/images/ into a single flat target_dir."""
    os.makedirs(target_dir, exist_ok=True)
    for i in range(1, 13):
        sub = os.path.join(base_dir, f'images_{i:03d}', 'images')
        if not os.path.isdir(sub):
            continue
        files = os.listdir(sub)
        print(f'  📦 images_{i:03d}/images/ → {len(files)} files')
        for fname in files:
            src, dst = os.path.join(sub, fname), os.path.join(target_dir, fname)
            if not os.path.exists(dst):
                shutil.move(src, dst)
    for i in range(1, 13):
        sub = os.path.join(base_dir, f'images_{i:03d}')
        if os.path.isdir(sub):
            shutil.rmtree(sub, ignore_errors=True)
    print(f'  ✅ Merged: {count_files(target_dir)} files')

# Strategy: a) Colab disk b) Drive merged c) Drive raw d) Kaggle download
n_colab = count_files(MERGED_DIR)
if n_colab >= MIN_IMAGES:
    print(f'✅ (a) Colab disk ready: {n_colab} files')
elif count_files(DRIVE_MERGED) >= MIN_IMAGES:
    print(f'📂 (b) Drive merged → copying to Colab (~5 min)...')
    os.makedirs(MERGED_DIR, exist_ok=True)
    for src_file in glob.glob(os.path.join(DRIVE_MERGED, '*')):
        dst_file = os.path.join(MERGED_DIR, os.path.basename(src_file))
        if not os.path.exists(dst_file):
            shutil.copy2(src_file, dst_file)
    print(f'   ✅ {count_files(MERGED_DIR)} files')
elif any(os.path.isdir(os.path.join(COLAB_DATA, f'images_{i:03d}')) for i in range(1, 13)):
    print('📦 (c) Colab raw subdirs → merge')
    merge_subdirs(COLAB_DATA, MERGED_DIR)
else:
    print(f'🌐 (d) Downloading from Kaggle: {KAGGLE_DATASET}')
    print('   ~45 GB — may take 10-20 minutes.')
    os.environ['KAGGLE_CONFIG_DIR'] = str(Path.home() / '.kaggle')
    subprocess.check_call([
        'kaggle', 'datasets', 'download',
        '-d', KAGGLE_DATASET,
        '-p', COLAB_DATA,
        '--unzip'
    ])
    # Kaggle download layout: COLAB_DATA/images_001..012/images/*.png
    print('🔧 Merging subdirs...')
    merge_subdirs(COLAB_DATA, MERGED_DIR)

# Copy the CSV into data/raw/
csv_src_candidates = [
    os.path.join(COLAB_DATA, CSV_NAME),
    os.path.join(DRIVE_DATA, CSV_NAME),
]
csv_dst = os.path.join(PROJECT_ROOT, 'data', 'raw', CSV_NAME)
os.makedirs(os.path.dirname(csv_dst), exist_ok=True)
for csv_src in csv_src_candidates:
    if os.path.exists(csv_src):
        shutil.copy2(csv_src, csv_dst)
        print(f'✅ CSV: {csv_src} → {csv_dst}')
        break
else:
    raise FileNotFoundError(f'{CSV_NAME} not found. Kaggle download incomplete.')

IMAGE_DIR = MERGED_DIR
print(f'\n📊 Final: {count_files(IMAGE_DIR):,} images @ {IMAGE_DIR}')

# Persist IMAGE_DIR for the preprocess script
os.makedirs(os.path.join(PROJECT_ROOT, 'data', 'processed'), exist_ok=True)
with open(os.path.join(PROJECT_ROOT, 'data', 'processed', 'image_dir.txt'), 'w') as f:
    f.write(IMAGE_DIR)

## 7. Preprocess + train/val/test/calibration split

In [ ]:
import pandas as pd

# preprocess.py produces train/val/test — we'll carve out calibration from val below
subprocess.check_call([
    sys.executable, 'scripts/preprocess.py',
    '--image-dir', IMAGE_DIR,
])

print('\n📊 Split statistics:')
for split in ['train', 'val', 'test']:
    p = f'data/processed/{split}.csv'
    if os.path.exists(p):
        df = pd.read_csv(p)
        label_col = 'Label' if 'Label' in df.columns else 'label'
        n_pos = (df[label_col] == 1).sum()
        n_neg = (df[label_col] == 0).sum()
        print(f'  {split:5s}: {len(df):6d} | Pneumo: {n_pos:5d} | Normal: {n_neg:5d} | 1:{n_neg/max(n_pos,1):.1f}')
    else:
        print(f'  ⚠️  {p} missing!')

# Calibration split (30% of val) — required by CalibrationService
calib_csv = 'data/processed/calibration.csv'
if not os.path.exists(calib_csv):
    val_df = pd.read_csv('data/processed/val.csv')
    calib_df = val_df.sample(frac=0.3, random_state=42)
    calib_df.to_csv(calib_csv, index=False)
    print(f'  ✅ Calibration split created: {len(calib_df)} samples')

## 8. Download Ark+ checkpoint (fallback: Swin-Base ImageNet)

In [ ]:
if RUN_TIER2_ARK:
    print('⬇️  Downloading Ark+ checkpoint (4 candidate URLs + Swin-Base fallback)...')
    rc = subprocess.call([sys.executable, 'scripts/download_ark_plus.py'])
    print(f'   Exit code: {rc}')
    print()
    subprocess.call(['ls', '-la', 'outputs/models/'])
else:
    print('⏭️  Ark+ skipped (RUN_TIER2_ARK=False).')

## 9. Train Tier 1 (MobileNetV2)

Output: `outputs/models/Tier1_MobileNetV2_Colab/best_model.pth`

In [ ]:
from application.dto.training_config_dto import TrainingConfigDTO
from application.services.training_service import TrainingService
import shutil

service = TrainingService()

if RUN_TIER1:
    tier1_cfg = TrainingConfigDTO(
        backbone='mobilenet_v2',
        run_name='Tier1_MobileNetV2_Colab',
        epochs=1 if DRY_RUN else 50,
        batch_size=32,
        lr_backbone=1e-4,
        lr_head=1e-3,
        early_stopping_patience=7,
        seed=42,
        use_synthetic=False,  # synthetic data disabled
    )
    print(f'🚀 Tier 1 training starting (epochs={tier1_cfg.epochs})...')
    tier1_result = service.train_model(tier1_cfg, dry_run=DRY_RUN)
    print(f'\n✅ Tier 1 done:')
    print(f'   Best Val AUC: {tier1_result["best_val_auc"]:.4f}')
    print(f'   Weights: {tier1_result["model_weight_path"]}')
    
    # alias to the filename expected by export_onnx.py
    legacy_path = 'outputs/models/best_tier1_mobilenet.pth'
    shutil.copy2(tier1_result['model_weight_path'], legacy_path)
    print(f'   Legacy alias: {legacy_path}')
else:
    print('⏭️  Tier 1 skipped.')

## 10. Train Tier 2 (EfficientNet-B4)

Output: `outputs/models/Tier2_EfficientNetB4_Colab/best_model.pth`

In [ ]:
if RUN_TIER2_EFFNET:
    tier2_eff_cfg = TrainingConfigDTO(
        backbone='efficientnet_b4',
        run_name='Tier2_EfficientNetB4_Colab',
        epochs=1 if DRY_RUN else 50,
        batch_size=16,
        lr_backbone=5e-5,
        lr_head=5e-4,
        early_stopping_patience=7,
        seed=42,
        use_synthetic=False,
    )
    print(f'🚀 Tier 2 EffNetB4 training starting (epochs={tier2_eff_cfg.epochs})...')
    tier2_eff_result = service.train_model(tier2_eff_cfg, dry_run=DRY_RUN)
    print(f'\n✅ Tier 2 EffNet done:')
    print(f'   Best Val AUC: {tier2_eff_result["best_val_auc"]:.4f}')
    print(f'   Weights: {tier2_eff_result["model_weight_path"]}')
    
    legacy_path = 'outputs/models/best_tier2_efficientnet.pth'
    shutil.copy2(tier2_eff_result['model_weight_path'], legacy_path)
    print(f'   Legacy alias: {legacy_path}')
else:
    print('⏭️  Tier 2 EffNet skipped.')

## 11. Train Tier 2 (Ark+ Swin)

Output: `outputs/models/Tier2_ArkPlus_Colab/best_model.pth`

In [ ]:
if RUN_TIER2_ARK:
    tier2_ark_cfg = TrainingConfigDTO(
        backbone='ark_plus',
        run_name='Tier2_ArkPlus_Colab',
        epochs=1 if DRY_RUN else 40,
        batch_size=12,
        lr_backbone=2e-5,
        lr_head=2e-4,
        early_stopping_patience=8,
        seed=42,
        use_synthetic=False,
    )
    print(f'🚀 Tier 2 Ark+ training starting (epochs={tier2_ark_cfg.epochs})...')
    tier2_ark_result = service.train_model(tier2_ark_cfg, dry_run=DRY_RUN)
    print(f'\n✅ Tier 2 Ark+ done:')
    print(f'   Best Val AUC: {tier2_ark_result["best_val_auc"]:.4f}')
    print(f'   Weights: {tier2_ark_result["model_weight_path"]}')
    
    legacy_path = 'outputs/models/best_tier2_arkplus.pth'
    shutil.copy2(tier2_ark_result['model_weight_path'], legacy_path)
    print(f'   Legacy alias: {legacy_path}')
else:
    print('⏭️  Tier 2 Ark+ skipped.')

## 12. Calibration — Temperature Scaling + ECE

Temperature scaling on Tier 2 EffNet, ECE computed on the validation set.

In [ ]:
if RUN_CALIBRATION:
    import torch
    import numpy as np
    from torch.utils.data import DataLoader
    from application.services.calibration_service import CalibrationService
    from core.models.factory import ModelFactory
    from infrastructure.data.dataset import NIHChestXrayDataset
    import core.models.tier2_efficientnet  # registration
    
    weights_path = 'outputs/models/best_tier2_efficientnet.pth'
    if not os.path.exists(weights_path):
        print(f'⚠️  {weights_path} missing — skipping calibration.')
    else:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model = ModelFactory.create('efficientnet_b4')
        state = torch.load(weights_path, map_location=device)
        # CheckpointObserver state_dict or raw weights
        if isinstance(state, dict) and 'model_state_dict' in state:
            model.load_state_dict(state['model_state_dict'])
        else:
            model.load_state_dict(state)
        model.to(device).eval()
        
        calib_ds = NIHChestXrayDataset(csv_file='data/processed/calibration.csv')
        calib_loader = DataLoader(calib_ds, batch_size=32, shuffle=False, num_workers=0)
        
        calib_service = CalibrationService()
        T = calib_service.calibrate_model(model, calib_loader, device)
        
        # ECE
        probs, labels = [], []
        with torch.no_grad():
            for batch in calib_loader:
                if isinstance(batch, (list, tuple)):
                    x, y = batch[0].to(device), batch[1]
                else:
                    x, y = batch['image'].to(device), batch['label']
                logits = model(x) / T
                p = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
                probs.append(p)
                labels.append(y.numpy() if hasattr(y, 'numpy') else np.array(y))
        probs = np.concatenate(probs)
        labels = np.concatenate(labels)
        ece = calib_service.calculate_ece(probs, labels, n_bins=10)
        
        # save temperature
        os.makedirs('outputs/results', exist_ok=True)
        torch.save({'temperature': T, 'ece': ece}, 'outputs/results/temperature.pt')
        
        # reliability diagram
        calib_service.save_reliability_diagram(probs, labels, filename='reliability_tier2_eff.png')
        
        print(f'✅ Calibration:')
        print(f'   Optimal T: {T:.4f}')
        print(f'   ECE @ T:   {ece:.4f}')
else:
    print('⏭️  Calibration skipped.')

## 13. Ablation Matrix (A8–A15)

In [ ]:
if RUN_ABLATION:
    from application.orchestration.ablation_runner import AblationRunner
    runner = AblationRunner()
    print(f'🔬 Ablation runs (dry_run={DRY_RUN})...')
    results = runner.run_all(dry_run=DRY_RUN)
    print(f'\n✅ Ablation: {len(results)} runs completed.')
    
    # honest ablation.json
    print('\n📝 Regenerating ablation.json from MLflow...')
    subprocess.check_call([sys.executable, 'scripts/build_ablation_json.py'])
    
    import json
    with open('outputs/results/ablation.json') as f:
        ab_data = json.load(f)
    real = sum(1 for r in ab_data if r.get('provenance') == 'mlflow_run')
    print(f'   Real (mlflow_run): {real}/{len(ab_data)}')
else:
    print('⏭️  Ablation skipped.')

## 14. Statistical Tests (DeLong + Bootstrap)

In [ ]:
if RUN_STAT_TESTS:
    subprocess.check_call([
        sys.executable, 'scripts/statistical_tests.py',
        '--output', 'outputs/results/statistical_comparison.csv',
    ])
    
    if os.path.exists('outputs/results/statistical_comparison.csv'):
        df = pd.read_csv('outputs/results/statistical_comparison.csv')
        print(f'\n✅ Statistical tests: {len(df)} comparisons')
        print(df.head(10).to_string())
else:
    print('⏭️  Stat tests skipped.')

## 15. ONNX + INT8 export (mobile-friendly)

In [ ]:
if RUN_ONNX_EXPORT:
    for model_name, weights in [
        ('tier1', 'outputs/models/best_tier1_mobilenet.pth'),
        ('tier2', 'outputs/models/best_tier2_efficientnet.pth'),
    ]:
        if not os.path.exists(weights):
            print(f'⚠️  {weights} missing — ONNX export for {model_name} skipped.')
            continue
        
        cmd = [sys.executable, 'scripts/export_onnx.py', '--model', model_name]
        if model_name == 'tier2':
            cmd.append('--quantize')
        print(f'📦 ONNX export: {" ".join(cmd)}')
        subprocess.check_call(cmd)
    
    print()
    subprocess.call(['ls', '-la', 'outputs/models/'])
else:
    print('⏭️  ONNX export skipped.')

## 16. Sync to Drive + download ZIP

In [ ]:
import shutil, datetime

TIMESTAMP = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
ZIP_NAME  = f'xray_outputs_{TIMESTAMP}.zip'
ZIP_PATH  = f'/content/{ZIP_NAME}'

# Drive sync
if USE_DRIVE and os.path.exists('/content/drive/MyDrive'):
    print(f'📤 Drive sync → {DRIVE_OUTPUT_DIR}/{TIMESTAMP}/')
    run_dir = os.path.join(DRIVE_OUTPUT_DIR, TIMESTAMP)
    os.makedirs(run_dir, exist_ok=True)
    for subdir in ['outputs', 'experiments']:
        src = os.path.join(PROJECT_ROOT, subdir)
        if os.path.exists(src):
            dst = os.path.join(run_dir, subdir)
            print(f'   {src} → {dst}')
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
    print('   ✅ Drive sync complete.')

# Build ZIP
print(f'\n📦 Building ZIP: {ZIP_NAME}')
shutil.make_archive(
    base_name=ZIP_PATH.replace('.zip', ''),
    format='zip',
    root_dir=PROJECT_ROOT,
    base_dir='outputs',
)
size_mb = os.path.getsize(ZIP_PATH) / 1e6
print(f'   ✅ {ZIP_PATH} ({size_mb:.1f} MB)')

# Trigger browser download
from google.colab import files
print(f'\n⬇️  Downloading to your browser...')
files.download(ZIP_PATH)
print('   ✅ Download started.')

## 17. Final sanity check

In [ ]:
print('📁 outputs/models/')
subprocess.call(['ls', '-la', 'outputs/models/'])
print('\n📁 outputs/results/')
subprocess.call(['ls', '-la', 'outputs/results/'])
print('\n💾 Disk usage:')
subprocess.call(['du', '-sh', 'outputs/', 'experiments/mlruns/'], stderr=subprocess.DEVNULL)
print('\n🎉 All done.')